### Coleta de dados de Temperatura

Será utilizado o dataset derived-era5-single-levels-daily-statistics do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=documentation

Os dados extraídos:
<pre>
- Temperatura   -> variável 2m_temperature, retorna a temperatura em Kelvin, será necessário uma conversão (subtrair -273,15)
</pre>

Os dados serão coletados por Ano e Mês 

Os dados requisitados estão no retangulo geográfico geográfico [6, -74, -34, -35] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil




In [2]:
import cdsapi
import sys, os
import xarray as xr
import dask.dataframe as dd
from pyspark.sql import functions as F

In [3]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


c:\Marco Conti\Projetos\mais_einstein\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)

c:\Marco Conti\Projetos\mais_einstein\Ondas_Calor


In [ ]:
import xarray as xr
import gcsfs

# 1. Conecta de forma anônima e rápida
fs = gcsfs.GCSFileSystem(token='anon')

# 2. Usa o bucket dedicado APENAS para variáveis de superfície (Single Levels)
mapper = fs.get_mapper('gs://gcp-public-data-arco-era5/co/single-levels-surface-one-level-reanalysis.zarr-v2')

# 3. Abre os metadados (carrega em segundos)
ds = xr.open_zarr(mapper, consolidated=True)

# 4. Seleciona APENAS a temperatura (t2m) e filtra os 30 anos
# Nota: O nome da variável pode ser '2m_temperature' ou 't2m'
t2m = ds['2m_temperature'].sel(time=slice('1994-01-01', '2023-12-31'))

# 5. O SEGREDO PARA FICAR LEVE: Faça o recorte espacial ANTES de carregar para a memória
# Exemplo para a América do Sul / Brasil (Lat: -35 a 5, Lon: -75 a -30 ou 285 a 330)
t2m_regiao = t2m.sel(
    latitude=slice(5, -35), 
    longitude=slice(285, 330) # No ERA5 algumas longitudes vão de 0 a 360
)

print(t2m_regiao)

In [ ]:
# t2m.to_netc;df('C:\\Marco Conti\\Projetos\\Dados\\ERA5_2023_t2m.nc')
type(t2m)

df = t2m.to_dataframe().reset_index()
print(type(df))
print(df.head())

In [7]:
def get_cdsapi_authentication():
    url = os.getenv("ECMWF_DATASTORES_URL")
    key = os.getenv("ECMWF_DATASTORES_KEY")
    return url, key

def get_t2m(year):
    dataset = "derived-era5-single-levels-daily-statistics"
    request = {
        "product_type": "reanalysis",
        "variable": ["2m_temperature"],
        "year": f"{year}",
        "month": [
            "01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"
        ],
        "day": [
            "01", "02", "03", "04", "05", "06", "07", "08", "09",
            "10", "11", "12", "13", "14", "15", "16", "17", "18",
            "19", "20", "21", "22", "23", "24", "25", "26", "27",
            "28", "29", "30", "31"
        ],
        "daily_statistic": "daily_mean",
        "time_zone": "utc-03:00",
        "frequency": "1_hourly",
        # Retangulo geográfico definido por Norte, Oeste, Sul e Leste em graus onde está o Brasil
        "area": [6      # Norte
                ,-74    # Oeste
                ,-34    # Sul
                ,-38]   # Leste          
    }

    # Informações de autenticação estão em:
    # C:\Users\DRT90628\.ecmwfdatastoresrc
    # *** Criar um novo contrato de autenticação deverá ser criado usando um usuário de serviços do Einstein
    url, key = get_cdsapi_authentication()

    client = \
        cdsapi.Client(url = url
                     ,key = key
        )

    ret_download = client.retrieve(dataset, request).download()

    os.rename(ret_download, f"{DATA_PATH_ROOT}\ERA5-temperaturas\ERA5_t2m_{year}.nc")

    return ret_download

def convert_t2m_dataset_to_spark_dataframe(project_path, ret_download ):

    with xr.open_dataset(f"{project_path}\{ret_download}"
                        ,engine="netcdf4"
                        ,chunks={"time": 365
                                ,"latitude": 100
                                ,"longitude": 100 }
                        ) as ds:

        # Transforma o Dataset em um Spark Dataframe
        df_dask        = ds.to_dask_dataframe()
        df_dask_c      = df_dask.compute()
        df_temperatura = spark.createDataFrame(df_dask_c)

    return df_temperatura

def convert_t2m_dataset_to_pandas_to_spark_dataframe(project_path, ret_download ):

    with xr.open_dataset(f"{project_path}\{ret_download}", engine="netcdf4") as ds:
        
        # Recorta a área do Brasil ANTES de chamar to_dataframe()
        # Verifica a orientação das longitudes no arquivo (-180 a 180 ou 0 a 360)
        lon_min, lon_max = (-74, -34) if ds.longitude.min() < 0 else (286, 326)
        
        ds_br = ds.sel(
            latitude=slice(6, -34),    # Norte para Sul
            longitude=slice(lon_min, lon_max)
        )
        
        # Agora a conversão cabe com folga na RAM
        pdf = ds_br.to_dataframe().dropna().reset_index()

        # Envia o DataFrame já reduzido e filtrado para o Spark
        df_temperatura = spark.createDataFrame(pdf)

        return df_temperatura

def transform_data(df_temperatura):
    drop_cols = ["valid_time", "t2m", "number"]

    df_temperatura_final = \
        (df_temperatura
            .withColumns({"data_medicao"    : F.col("valid_time").cast("date")
                         ,"indicador"       : F.lit("temperatura") 
                         ,"valor"           : (F.col("t2m") - F.lit(273.15)).cast("double") # Converte a temperatura de Kelvin para Celsius
                         ,"unidade_medida"  : F.lit("celsius")})
            .drop(*drop_cols)
    )

    df_temperatura_final = \
        (df_temperatura_final
            .select("data_medicao"
                   ,"latitude"
                   ,"longitude"
                   ,"indicador"
                   ,"valor"
                   ,"unidade_medida"))

    return df_temperatura_final

def write_data(df_temperatura_final, write_path, ano):
    # df_temperatura_final.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_t2m_temperatura.csv", index=False)

    df_temperatura_final.toPandas().to_parquet(f"{write_path}\\ERA5-temperaturas\\{ano}\\ERA5_temperatura.parquet")

def remove_aux_file(file_name):
    os.remove(file_name)


In [8]:

# Converte os dados de temperatura para um Spark Dataframe

path            = r"C:\Users\DRT90628\Downloads"
ret_download    = "e5.oper.an.sfc.128_015_aluvp.ll025sc.2025120100_2025123123.nc"


df_temperatura  = convert_t2m_dataset_to_pandas_to_spark_dataframe(path, ret_download)

# # Converte a temperatura de Kelsin para Celsius e adiciona coluna de unidade de medida
# df_temperatura_final = transform_data(df_temperatura)

# # Escreve os dados em formato parquet
# write_data(df_temperatura_final, DATA_PATH_ROOT, 2026)

MemoryError: Unable to allocate 36.8 MiB for an array with shape (19285224,) and data type int16

In [ ]:
df_temperatura.printSchema()
df_temperatura.limit(10).show(truncate = False)

In [ ]:
from datetime import datetime 

years_process = [1996, 1997, 1998, 1999
                ,2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009
                ,2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]

# years_process = [1990]
for year in years_process:

    start = datetime(2026, 7, 29).now()
    print("Start download - year : ",year, " - ", start, end="" )

    # Faz download do arquivo de temperaturas do portal Copernicus
    # retorno em formato .nc -> NetCDF (Network Common Data Form)
    ret_download         = get_t2m(year)

    # # Converte os dados de temperatura para um Spark Dataframe
    # df_temperatura       = convert_t2m_dataset_to_spark_dataframe(PROJECT_PATH, ret_download)

    # # Converte a temperatura de Kelsin para Celsius e adiciona coluna de unidade de medida
    # df_temperatura_final = transform_data(df_temperatura)

    # # Escreve os dados em formato parquet
    # write_data(df_temperatura_final, DATA_PATH_ROOT, year)

    # remove_aux_file(ret_download)

    finish = datetime(2026, 7, 29).now()

    print(f" - Download completed: {ret_download} - {finish} - {(finish - start)} \n")

Converte os dados baixados do ERA5, que estão em formato NetCDF, para um dataset (xarray.core.dataset.Dataset)

In [ ]:
ret_download = "b80f05d1220d008d67fbe12abee7958d.nc" # dados de 2023
# ret_download = "d03a86fd89e8c00cdf3bea8d8fb499f.nc" # dados de 2024
# ret_download = "507e00f8fa0894949bc892e52ab7b3b7.nc" # dados de 2025
path = f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{ret_download}"
print(path)

Renomeia nome de colunas e converte o valor da Temperatura recebida do ERA5 está em Kelvin, para converter para Celsius, subtrair 273.15

In [ ]:
df_temperatura.filter("latitude = -34.0 and longitude = -67.0").orderBy("t2m").show(10,False)

In [ ]:
drop_cols = ["valid_time", "t2m", "number"]

df_temperatura_final = \
    (df_temperatura
        .withColumns({"data_medicao": F.col("valid_time").cast("date")
                     ,"indicador": F.lit("temperatura") 
                     ,"valor": (F.col("t2m") - F.lit(273.15)).cast("double")
                     ,"unidade_medida": F.lit("celsius")})
         .drop(*drop_cols)
    )

df_temperatura_final = \
    (df_temperatura_final
        .select("data_medicao"
               ,"latitude"
               ,"longitude"
               ,"indicador"
               ,"valor"
               ,"unidade_medida"))

df_temperatura_final.printSchema()

df_temperatura_final.show()

In [ ]:
# df_temperatura_final.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_t2m_temperatura.csv", index=False)

df_temperatura_final.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\mais_einstein\\dados\\ERA5-temperaturas\\2025\\ERA5_temperatura.parquet")
